# ДЗ 7.2. Стержень из двух материалов

**Константин Мурусидзе, 6 курс мехмата МГУ · уровень 🟡**

Нахожу распределение температуры и теплового потока в составном стержне.
Проверяю стационарное решение и тепловой баланс, затем сравниваю
$(k u_x)_x$ с $k u_{xx}$ при разных отношениях теплопроводностей.

[Условия курса, занятие 7](https://github.com/yartmihant/Students6Year2026/blob/main/README.md#занятие-7-параболические-уравнения-уравнение-теплопроводности).

## Модель и разностные схемы

На $0\le x\le1$ решаю $u_t=(ku_x)_x$, где $k=1$ слева от $x=0.5$
и $k=1/R$ справа. Здесь $R=k_{\rm left}/k_{\rm right}$, объёмная
теплоёмкость равна 1. На концах $u(0,t)=1$ и $u(1,t)=0$;
$u(x,0)=1-x$. Тепловой поток $q=-ku_x$ положителен вправо.
В идеальном контакте температура и поток непрерывны.

На сетке из $N$ ячеек ширины $h=1/N$ храню температуры $U_i$ в центрах.
$N$ чётное, стык лежит на грани. Поток через внутреннюю грань вычисляю
по суммарному сопротивлению двух полуячеек:

$$
q_{i+1/2}=g_{i+1/2}(U_i-U_{i+1}),\qquad
g_{i+1/2}=\left(\frac{h}{2k_i}+\frac{h}{2k_{i+1}}\right)^{-1}.
$$

Величина $h g_{i+1/2}$ — гармоническое среднее теплопроводностей.
У концов стержня сопротивление равно $h/(2k)$, так как заданная
температура находится на расстоянии $h/2$ от центра крайней ячейки.
Неявный Эйлер использует потоки с нового слоя:

$$
\frac{U_i^{n+1}-U_i^n}{\Delta t}
=\frac{q_{i-1/2}^{n+1}-q_{i+1/2}^{n+1}}{h}.
$$

В сравнительном расчёте заменяю правую часть на
$k_i(U_{i-1}-2U_i+U_{i+1})/h^2$. На внешних границах задаю
фиктивные значения $2u_{\rm boundary}-U_i$, чтобы среднее с соседним
центром давало нужную граничную температуру. На стыке дополнительное
условие для этого оператора не ставлю. Интегратор по времени тот же.

Использую $N=50,100,200$ и $\Delta t=h$. Для всех вариантов сравниваю
профили в одни и те же физические моменты, включая $t=80$.
Критерий установления: $\|U^{n+1}-U^n\|_\infty/\Delta t<10^{-10}$.
После его выполнения продолжаю расчёт до общего конечного времени,
чтобы сравнить накопленные потоки.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, eye
from scipy.sparse.linalg import splu
from IPython.display import Markdown, display

plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.25})
grids = (50, 100, 200)
ratios = (1, 10, 100)
t_end = 80.0
snapshot_times = (0.0, 0.2, 1.0, 5.0, t_end)
steady_tol = 1e-10
left_bc, right_bc = 1.0, 0.0
labels = {"flux": r"$(ku_x)_x$", "central": r"$ku_{xx}$"}
colors = {"flux": "#24609f", "central": "#b95c18"}
ratio_colors = {1: "#7b8795", 10: "#24609f", 100: "#19765b"}


def table(headers, rows):
    def row_text(row):
        return "| " + " | ".join(str(item).replace("|", r"\|") for item in row) + " |"
    lines = [row_text(headers),
             "| " + " | ".join(["---"]*len(headers)) + " |"]
    lines.extend(row_text(row) for row in rows)
    display(Markdown("\n".join(lines)))


def conductance(k, h):
    half_resistance = 0.5*h/k
    return 1 / np.r_[half_resistance[0],
                      half_resistance[:-1]+half_resistance[1:],
                      half_resistance[-1]]


def operator(k, h, model):
    rhs_bc = np.zeros(len(k))
    if model == "flux":
        g = conductance(k, h)
        diag = -(g[:-1]+g[1:])/h
        lower = upper = g[1:-1]/h
        rhs_bc[0] = g[0]*left_bc/h
        rhs_bc[-1] = g[-1]*right_bc/h
    else:
        a = k/h**2
        diag = -2*a
        diag[0] -= a[0]
        diag[-1] -= a[-1]
        lower, upper = a[1:], a[:-1]
        rhs_bc[0] = 2*a[0]*left_bc
        rhs_bc[-1] = 2*a[-1]*right_bc
    return diags([lower, diag, upper], [-1, 0, 1], format="csc"), rhs_bc


def face_fluxes(u, k, h, model):
    jumps = np.r_[left_bc-u[0], u[:-1]-u[1:], u[-1]-right_bc]
    if model == "flux":
        q = conductance(k, h)*jumps
        return q, q
    # На стыке одна разность температур даёт два односторонних потока.
    gradient = jumps/h
    gradient[[0, -1]] *= 2
    return np.r_[k[0], k]*gradient, np.r_[k, k[-1]]*gradient


def solve(n, ratio, model):
    h = 1/n
    dt = h
    n_steps = round(t_end/dt)
    time = dt*np.arange(n_steps+1)
    x = h*(np.arange(n)+0.5)
    k = np.where(x < 0.5, 1.0, 1.0/ratio)
    a, bc = operator(k, h, model)
    # При фиксированном шаге факторизация нужна только один раз.
    solve_layer = splu(eye(n, format="csc")-dt*a, permc_spec="NATURAL").solve
    u = 1-x
    heat = np.empty(n_steps+1)
    q_in = np.empty_like(heat)
    q_out = np.empty_like(heat)
    snapshot_indices = {round(t/dt): t for t in snapshot_times}
    snapshots = {}
    steady_time = None
    rate = np.max(np.abs(a @ u + bc))

    for j, t in enumerate(time):
        if j:
            next_u = solve_layer(u + dt*bc)
            rate = np.max(np.abs(next_u-u))/dt
            u = next_u
        if steady_time is None and rate < steady_tol:
            steady_time = t
        heat[j] = h*np.sum(u)
        q_in[j] = 2*k[0]*(left_bc-u[0])/h
        q_out[j] = 2*k[-1]*(u[-1]-right_bc)/h
        if j in snapshot_indices:
            snapshots[snapshot_indices[j]] = u.copy()

    in_total = np.r_[0.0, np.cumsum(dt*q_in[1:])]
    out_total = np.r_[0.0, np.cumsum(dt*q_out[1:])]
    return {
        "x": x, "h": h, "k": k, "time": time, "snapshots": snapshots,
        "q": face_fluxes(u, k, h, model), "H": heat,
        "in": in_total, "out": out_total, "B": heat-heat[0]-in_total+out_total,
        "steady_time": steady_time, "final_rate": rate,
    }


def steady_exact(x, ratio):
    q = 2/(1+ratio)
    resistance = np.minimum(x, 0.5) + ratio*np.maximum(x-0.5, 0.0)
    return 1-q*resistance, q

## Установление при отношении теплопроводностей 10 · 🟢

В стационарном режиме поток постоянен, поэтому

$$
q_*=\left(\tfrac12+\tfrac{R}{2}\right)^{-1},\qquad
u_*(x)=1-q_*\bigl[\min(x,0.5)+R\max(x-0.5,0)\bigr].
$$

Сравниваю температуры в центрах ячеек, а потоки — на всех гранях.
Здесь оба материала занимают целое число ячеек. Стационарный кусочно
линейный профиль точно согласован с формулой сопротивлений, поэтому
остаточные ошибки определяются установлением и округлением.

In [ ]:
results = {}
rows = []
for n in grids:
    result = results[n, 10, "flux"] = solve(n, 10, "flux")
    u_star, q_star = steady_exact(result["x"], 10)
    error_u = np.max(np.abs(result["snapshots"][t_end]-u_star))
    error_q = np.max(np.abs(result["q"][0]-q_star))
    rows.append((n, f'{result["h"]:.4f}', f'{result["steady_time"]:.3f}',
                 f"{error_u:.3e}", f"{error_q:.3e}"))
table(("N", "h = Δt", "Время установления", "max |u−u*|", "max |q−q*|"), rows)

result = results[100, 10, "flux"]
x_dense = np.linspace(0, 1, 501)
u_star, q_star = steady_exact(x_dense, 10)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4), layout="constrained")
for t in snapshot_times:
    axes[0].plot(np.r_[0, result["x"], 1], np.r_[1, result["snapshots"][t], 0],
                 label=f"t = {t:g}")
axes[0].plot(x_dense, u_star, "k--", lw=1.5, label="Точное стационарное")
axes[0].set(xlabel="x", ylabel="u", title="Установление температуры")
axes[0].legend(fontsize=8, loc="lower left")
axes[1].plot(np.linspace(0, 1, 101), result["q"][0], color=colors["flux"], label="Численный")
axes[1].axhline(q_star, color="black", ls="--", label=f"Точный q* = {q_star:.5f}")
axes[1].set(xlabel="x", ylabel="q", title="Поток при t = 80", ylim=(0, 1.4*q_star))
axes[1].legend(loc="lower right", fontsize=8)
for ax in axes:
    ax.axvline(0.5, color="gray", alpha=0.5, lw=1)
fig.suptitle("Стержень с отношением теплопроводностей 10, N = 100")
plt.show()

В стационарном состоянии один и тот же поток проходит через оба
материала. Поэтому $|u_x|=q_*/k$: в правой половине с $k=0.1$
градиент по модулю в десять раз больше, чем в левой.

## Тепловой баланс и сравнение операторов · 🟡

Дискретное количество тепла $H^n=h\sum_i U_i^n$.
Если сложить уравнения консервативной схемы по ячейкам, внутренние
потоки сократятся. Останутся только вход и выход:

$$
H^n-H^0=I_{\rm in}^n-I_{\rm out}^n,\qquad
I_{\rm in/out}^n=\sum_{m=1}^n \Delta t\,q_{\rm in/out}^{m}.
$$

Суммирую потоки именно на новых слоях, как в неявном Эйлере.
Для обоих операторов рассчитываю невязку
$B=H-H^0-I_{\rm in}+I_{\rm out}$ с одинаковым правилом вычисления
потоков на концах. Для $ku_{xx}$ показываю два значения потока
на стыке: со стороны левого и правого материала.

In [ ]:
for n in grids:
    for ratio in ratios:
        for model in labels:
            key = (n, ratio, model)
            if key not in results:
                results[key] = solve(n, ratio, model)

n = 100
fig, axes = plt.subplots(2, 3, figsize=(12, 6.1), sharex=True, layout="constrained")
faces = np.linspace(0, 1, n+1)
middle = n//2
for col, ratio in enumerate(ratios):
    for model, style in [("flux", "-"), ("central", "--")]:
        result = results[n, ratio, model]
        axes[0, col].plot(np.r_[0, result["x"], 1], np.r_[1, result["snapshots"][t_end], 0],
                         style, color=colors[model], lw=1.7, label=labels[model])
        q_left, q_right = result["q"]
        axes[1, col].plot(faces[:middle+1], q_left[:middle+1], style,
                         color=colors[model], lw=1.7, label=labels[model])
        axes[1, col].plot(faces[middle:], q_right[middle:], style,
                         color=colors[model], lw=1.7)
        if model == "central":
            axes[1, col].plot([0.5, 0.5], [q_left[middle], q_right[middle]], "o:",
                             color=colors[model], ms=4)
    axes[0, col].set(title=f"Отношение k: {ratio}:1", ylim=(-0.03, 1.04))
    axes[1, col].set(xlabel="x", ylim=(-0.03, 1.1))
for ax in axes.flat:
    ax.axvline(0.5, color="gray", lw=0.8, alpha=0.5)
axes[0, 0].set_ylabel("Температура u")
axes[1, 0].set_ylabel("Поток q")
axes[0, -1].legend(loc="lower left")
axes[1, -1].legend(loc="center right")
fig.suptitle("Температура и поток при t = 80, N = 100")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11.8, 7), layout="constrained")
for ratio in ratios:
    for model, style in [("flux", "-"), ("central", "--")]:
        result = results[n, ratio, model]
        time = result["time"]
        # Общий шаг отображения 0.05; интегрирование использует Δt=h.
        idx = np.arange(0, len(time), 5)
        t_plot = time[idx]
        name = f"R={ratio}, {labels[model]}"
        axes[0, 0].plot(t_plot, result["H"][idx], style,
                        color=ratio_colors[ratio], label=name)
        if model == "flux":
            axes[1, 0].plot(t_plot, result["B"][idx], color=ratio_colors[ratio],
                            label=f"R={ratio}")
        else:
            axes[1, 1].plot(t_plot, result["B"][idx], color=ratio_colors[ratio],
                            label=f"R={ratio}")
        if ratio == 10:
            for key, label, color in [("in", "Приток", "#24609f"),
                                      ("out", "Отток", "#b95c18")]:
                axes[0, 1].plot(t_plot, result[key][idx], style, color=color,
                                label=f"{label}, {labels[model]}")
axes[0, 0].set(title="Количество тепла", xlim=(0, 15), ylabel="H")
axes[0, 1].set(title="Накопленные потоки при R=10", ylabel="Количество тепла")
axes[1, 0].set(title=r"Невязка баланса для $(ku_x)_x$", ylabel="B")
axes[1, 0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useOffset=False)
axes[1, 1].set(title=r"Невязка баланса для $ku_{xx}$", ylabel="B")
for ax in axes.flat:
    ax.set_xlabel("t")
    ax.legend(fontsize=8)
plt.show()

## Проверка при сгущении сетки

Сравниваю профили в центрах ячеек в момент $t=80$.
Скачок потока измеряю как $q(0.5+)-q(0.5-)$.
В консервативной схеме на стыке один общий поток, так что скачок
равен нулю. В таблице также приведены конечная невязка для $ku_{xx}$
и максимальная по времени невязка консервативной схемы.

Для проверки в переходном процессе сопоставляю также консервативные
профили при $t=1$. Решение на $N=200$ усредняю по ячейкам более грубой
сетки и сравниваю с её собственным расчётом.

In [ ]:
rows = []
for ratio in ratios:
    for n in grids:
        correct = results[n, ratio, "flux"]
        central = results[n, ratio, "central"]
        diff = np.max(np.abs(correct["snapshots"][t_end]-central["snapshots"][t_end]))
        q_left, q_right = central["q"]
        jump = q_right[n//2]-q_left[n//2]
        rows.append((ratio, n, f"{diff:.5f}", f"{jump:.5f}", f'{central["B"][-1]:.3e}',
                     f'{np.max(np.abs(correct["B"])):.2e}'))
table(("R", "N", "max |u₁−u₂|", "Скачок q у kuₓₓ",
       "B(80) у kuₓₓ", "max |B| у (kuₓ)ₓ"), rows)

rows = []
for ratio in (10, 100):
    fine = results[200, ratio, "flux"]["snapshots"][1.0]
    errors = []
    for n in (50, 100):
        restricted = fine.reshape(n, 200//n).mean(axis=1)
        coarse = results[n, ratio, "flux"]["snapshots"][1.0]
        errors.append(np.max(np.abs(coarse-restricted)))
    rows.append((ratio, *(f"{e:.3e}" for e in errors)))
table(("R", "Разность N=50 и N=200 при t=1", "Разность N=100 и N=200 при t=1"), rows)

## Выводы

При $R=1$ операторы совпадают. При $R>1$ консервативная схема меняет
начальный профиль: слева температура падает слабо, справа — круто.
Поток на стыке остаётся непрерывным, а тепловой баланс выполняется
с точностью численного решения.

Для $ku_{xx}$ начальный профиль $1-x$ уже стационарен: вторая разность
равна нулю, в том числе около стыка. Температура не меняется, но наклон
$-1$ даёт поток $1$ слева и $1/R$ справа. Эти потоки не совпадают
на стыке. Поэтому при сложении по ячейкам остаётся вклад
$q(0.5+)-q(0.5-)=1/R-1$, и невязка физического теплового баланса
растёт по модулю. Например, при $R=10$ получается $B(80)\approx-72$.

Сгущение сетки не устраняет это различие: при $R=10$ разность профилей
стремится к $0.40909$, при $R=100$ — к $0.49010$. Это видно и из
температур на стыке: для консервативной модели $u_*(0.5)=R/(1+R)$,
для второго оператора $u(0.5)=0.5$. Причина расхождения — разные
уравнения, а не недостаточное разрешение сетки.